# Day 53 — MLOps intro: experiment tracking with MLflow
Objectives:
- Track params, metrics, and artifacts with MLflow.
- Organize experiments and runs.
- Save and load models from MLflow.
Note: `pip install mlflow` (already in requirements.txt). By default this uses a local `mlruns` folder.

In [ ]:
import mlflow
import mlflow.sklearn
from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score
X,y = load_breast_cancer(return_X_y=True)
Xtr,Xte,ytr,yte = train_test_split(X,y, test_size=0.2, random_state=42, stratify=y)
pipe = Pipeline([('sc', StandardScaler()), ('clf', LogisticRegression(max_iter=1000))])
mlflow.set_experiment('ds-60day-bc-experiment')
with mlflow.start_run(run_name='baseline-logreg'):
    mlflow.log_param('model', 'LogisticRegression')
    mlflow.log_param('scale', True)
    pipe.fit(Xtr,ytr)
    yprob = pipe.predict_proba(Xte)[:,1]
    auc = roc_auc_score(yte, yprob)
    mlflow.log_metric('roc_auc', auc)
    mlflow.sklearn.log_model(pipe, artifact_path='model')
    print('ROC AUC:', auc)


## Viewing results
Run the MLflow UI in a terminal:
```bash
mlflow ui --backend-store-uri mlruns
```
Then open http://127.0.0.1:5000 to view experiments.

## Loading a model from MLflow
You can load the saved model artifact and use it for inference.

In [ ]:
# Example: load the last logged model (adjust run_id/artifact URI as needed)
# model_uri = 'runs:/<run_id>/model'
# loaded = mlflow.sklearn.load_model(model_uri)
# loaded.predict(Xte[:5])


## Exercises
1) Log additional params (e.g., C for LogisticRegression) and compare runs.
2) Add confusion matrix PNG as an artifact (matplotlib savefig + mlflow.log_artifact).
3) Try a different classifier (e.g., RandomForest) and compare AUC.
